# Olist E-Commerce — Data Exploration Notebook

## Objectives

This notebook answers 5 foundational questions before designing the Data Warehouse:

1. **What is the Fact Table?** → Identify the central transaction table
2. **What is the Grain?** → Define what one row in FactSales represents
3. **Where does Revenue come from?** → Clarify the revenue source (`payment_value` vs `price`)
4. **What are the Dimensions?** → Identify supporting dimension tables
5. **Do the data support all 17 Business Questions?** → Validate coverage per BQ

It also performs Data Profiling and Data Quality Assessment across all 9 source tables.

## 1. Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "../data/raw"

In [2]:
customers          = pd.read_csv(f"{DATA_PATH}/olist_customers_dataset.csv")
orders             = pd.read_csv(f"{DATA_PATH}/olist_orders_dataset.csv")
order_items        = pd.read_csv(f"{DATA_PATH}/olist_order_items_dataset.csv")
payments           = pd.read_csv(f"{DATA_PATH}/olist_order_payments_dataset.csv")
reviews            = pd.read_csv(f"{DATA_PATH}/olist_order_reviews_dataset.csv")
products           = pd.read_csv(f"{DATA_PATH}/olist_products_dataset.csv")
sellers            = pd.read_csv(f"{DATA_PATH}/olist_sellers_dataset.csv")
geolocation        = pd.read_csv(f"{DATA_PATH}/olist_geolocation_dataset.csv")
category_translation = pd.read_csv(f"{DATA_PATH}/product_category_name_translation.csv")

tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation,
}

print("All tables loaded successfully.")

All tables loaded successfully.


## 2. Dataset Overview

In [3]:
overview = pd.DataFrame({
    "table":   list(tables.keys()),
    "rows":    [df.shape[0] for df in tables.values()],
    "columns": [df.shape[1] for df in tables.values()],
})
overview

,table,rows,columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,sellers,3095,4
7,geolocation,1000163,5
8,category_translation,71,2


## 3. Data Profiling

Profile every source table: shape, columns, missing values, duplicates, and sample rows.  
This section is required to validate data quality before ETL design.

In [4]:
def explore_table(df, name):
    print("=" * 60)
    print(f"TABLE: {name.upper()}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("\nColumns:", df.columns.tolist())
    null_counts = df.isnull().sum()
    if null_counts.any():
        print("\nMissing Values:")
        print(null_counts[null_counts > 0])
    else:
        print("\nMissing Values: None")
    dup = df.duplicated().sum()
    print(f"\nDuplicates: {dup:,}")
    display(df.head(3))
    print()

In [5]:
explore_table(customers, "customers")

TABLE: CUSTOMERS
Shape: 99,441 rows × 5 columns

Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Missing Values: None

Duplicates: 0


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


In [6]:
explore_table(orders, "orders")

TABLE: ORDERS
Shape: 99,441 rows × 8 columns

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Missing Values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Duplicates: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [7]:
explore_table(order_items, "order_items")

TABLE: ORDER_ITEMS
Shape: 112,650 rows × 7 columns

Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Missing Values: None

Duplicates: 0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


In [8]:
explore_table(payments, "payments")

TABLE: PAYMENTS
Shape: 103,886 rows × 5 columns

Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Missing Values: None

Duplicates: 0


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [9]:
explore_table(reviews, "reviews")

TABLE: REVIEWS
Shape: 99,224 rows × 7 columns

Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Missing Values:
review_comment_title      87656
review_comment_message    58247
dtype: int64

Duplicates: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


In [10]:
explore_table(products, "products")

TABLE: PRODUCTS
Shape: 32,951 rows × 9 columns

Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Missing Values:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Duplicates: 0


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


In [11]:
explore_table(sellers, "sellers")

TABLE: SELLERS
Shape: 3,095 rows × 4 columns

Columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Missing Values: None

Duplicates: 0


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


In [12]:
explore_table(geolocation, "geolocation")

TABLE: GEOLOCATION
Shape: 1,000,163 rows × 5 columns

Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Missing Values: None

Duplicates: 261,831


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


In [13]:
explore_table(category_translation, "category_translation")

TABLE: CATEGORY_TRANSLATION
Shape: 71 rows × 2 columns

Columns: ['product_category_name', 'product_category_name_english']

Missing Values: None

Duplicates: 0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## 4. Data Quality Assessment

In [14]:
quality_report = pd.DataFrame({
    "table": list(tables.keys()),
    "rows": [df.shape[0] for df in tables.values()],
    "missing_values": [df.isnull().sum().sum() for df in tables.values()],
    "missing_pct": [
        round(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2)
        for df in tables.values()
    ],
    "duplicates": [df.duplicated().sum() for df in tables.values()],
})
quality_report

,table,rows,missing_values,missing_pct,duplicates
0,customers,99441,0,0.00,0
1,orders,99441,4908,0.62,0
2,order_items,112650,0,0.00,0
3,payments,103886,0,0.00,0
4,reviews,99224,145903,21.01,0
5,products,32951,2448,0.83,0
6,sellers,3095,0,0.00,0
7,geolocation,1000163,0,0.00,261831
8,category_translation,71,0,0.00,0


**Key Findings:**

1. `products` — missing values in `product_category_name`, `product_name_length`, `product_description_length`, `product_photos_qty`, and weight/dimension columns. Handle in ETL: fill with `UNKNOWN` / 0.
2. `reviews` — missing `review_comment_title` and `review_comment_message` (optional fields, not critical for KPIs).
3. `orders` — missing `order_delivered_customer_date` and `order_approved_at` for non-delivered orders (expected). Filter to `status = 'delivered'` for KPI calculations.
4. `geolocation` — duplicate `geolocation_zip_code_prefix` rows (multiple lat/lng per zip). Deduplicate by keeping the first occurrence per zip code in ETL.

## 5. Relationship Validation

Verify referential integrity between all source tables before building the star schema.

In [15]:
# customers ↔ orders
n_customers = customers["customer_id"].nunique()
n_orders    = orders["customer_id"].nunique()
print(f"Unique customer_ids in customers : {n_customers:,}")
print(f"Unique customer_ids in orders    : {n_orders:,}")
print(f"Orders with no matching customer : {orders[~orders['customer_id'].isin(customers['customer_id'])].shape[0]:,}")

Unique customer_ids in customers : 99,441
Unique customer_ids in orders    : 99,441
Orders with no matching customer : 0


In [16]:
# How many orders per unique customer (person)?
# Must use customer_unique_id — the true person-level identifier.
orders_with_uid = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id"
)
orders_per_customer = orders_with_uid.groupby("customer_unique_id").size()
print("Orders per unique customer:")
print(orders_per_customer.describe())
print(f"\nCustomers with >1 order: {(orders_per_customer > 1).sum():,}")

Orders per unique customer:
count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
dtype: float64

Customers with >1 order: 2,997


In [17]:
# orders ↔ order_items
unmatched_items = order_items[~order_items["order_id"].isin(orders["order_id"])]
print(f"order_items rows with no matching order_id: {len(unmatched_items):,}")

items_per_order = order_items.groupby("order_id").size()
print("\nItems per order:")
print(items_per_order.describe())

order_items rows with no matching order_id: 0

Items per order:
count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64


In [18]:
# orders ↔ payments  (one order can have multiple payment methods)
unmatched_pay = payments[~payments["order_id"].isin(orders["order_id"])]
print(f"payments rows with no matching order_id: {len(unmatched_pay):,}")

pay_per_order = payments.groupby("order_id").size()
print("\nPayment rows per order (split payments):")
print(pay_per_order.value_counts().head(5))

payments rows with no matching order_id: 0

Payment rows per order (split payments):
1    96479
2     2382
3      301
4      108
5       52
Name: count, dtype: int64


In [19]:
# orders ↔ reviews
unmatched_rev = reviews[~reviews["order_id"].isin(orders["order_id"])]
print(f"reviews rows with no matching order_id: {len(unmatched_rev):,}")

# order_items ↔ products
unmatched_prod = order_items[~order_items["product_id"].isin(products["product_id"])]
print(f"order_items rows with no matching product_id: {len(unmatched_prod):,}")

# order_items ↔ sellers
unmatched_sell = order_items[~order_items["seller_id"].isin(sellers["seller_id"])]
print(f"order_items rows with no matching seller_id: {len(unmatched_sell):,}")

reviews rows with no matching order_id: 0
order_items rows with no matching product_id: 0
order_items rows with no matching seller_id: 0


## 6. Q1 — What is the Fact Table?

The central transaction table is `order_items`.  
Each row represents **one product sold in one order** — it links to orders, products, sellers, and is the source for price and freight cost.

In [20]:
print("order_items columns:")
print(order_items.columns.tolist())
print()
print("Sample row:")
display(order_items.head(3))

order_items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Sample row:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


**Answer — 3 Fact Tables (final schema):**

> **Fan-out join problem:** `payment_value` cannot live in FACT_SALES.
> An order with 2 items would duplicate `payment_value` across 2 rows → `SUM()` double-counts.
> Solution: separate the 3 different grains into 3 fact tables.

---

**FACT_SALES** — Grain: 1 product in 1 order

| Column | Type | Source |
|---|---|---|
| `sales_key` | PK (surrogate) | generated |
| `order_id` | degenerate dim | orders |
| `order_item_id` | degenerate dim | order_items |
| `customer_key` | FK → DIM_CUSTOMER | customers.customer_unique_id |
| `product_key` | FK → DIM_PRODUCT | order_items.product_id |
| `seller_key` | FK → DIM_SELLER | order_items.seller_id |
| `date_key` | FK → DIM_DATE | orders.order_purchase_timestamp |
| `price` | measure | order_items |
| `freight_value` | measure | order_items |

---

**FACT_PAYMENTS** — Grain: 1 payment record per order

| Column | Type | Source |
|---|---|---|
| `payment_fact_key` | PK (surrogate) | generated |
| `order_id` | degenerate dim | order_payments |
| `customer_key` | FK → DIM_CUSTOMER | via orders → customers.customer_unique_id |
| `date_key` | FK → DIM_DATE | orders.order_purchase_timestamp |
| `payment_key` | FK → DIM_PAYMENT | order_payments.payment_type |
| `payment_sequential` | attribute | order_payments |
| `payment_value` | measure | order_payments |
| `payment_installments` | measure | order_payments |

---

**FACT_ORDER_EXPERIENCE** — Grain: 1 order

| Column | Type | Source |
|---|---|---|
| `experience_fact_key` | PK (surrogate) | generated |
| `order_id` | degenerate dim | orders |
| `customer_key` | FK → DIM_CUSTOMER | customers.customer_unique_id |
| `date_key` | FK → DIM_DATE | orders.order_purchase_timestamp |
| `delivery_days` | measure | derived: delivered − purchased |
| `delay_days` | measure | derived: delivered − estimated |
| `review_score` | measure | order_reviews |
| `is_late_delivery` | attribute (0/1) | derived: delay_days > 0 |

## 7. Q2 — What is the Grain?

In [21]:
# Verify: is (order_id, order_item_id) a unique key?
grain_check = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
print(f"Duplicate (order_id, order_item_id) pairs: {grain_check:,}")
print()
print(f"Total rows in order_items : {order_items.shape[0]:,}")
print(f"Unique order_id           : {order_items['order_id'].nunique():,}")
print(f"Unique product_id         : {order_items['product_id'].nunique():,}")
print(f"Unique seller_id          : {order_items['seller_id'].nunique():,}")

Duplicate (order_id, order_item_id) pairs: 0

Total rows in order_items : 112,650
Unique order_id           : 98,666
Unique product_id         : 32,951
Unique seller_id          : 3,095


**Answer — Grain:**

> **One row in FACT_SALES = one product line item in one order.**
>
> An order with 3 different products = 3 rows in FACT_SALES.  
> The unique key is `(order_id, order_item_id)`.

## 8. Q3 — Where does Revenue come from?

Two candidates: `order_items.price` and `order_payments.payment_value`.  
We need to compare both and justify the choice.

In [22]:
price_total   = order_items["price"].sum()
freight_total = order_items["freight_value"].sum()
payment_total = payments["payment_value"].sum()

print(f"SUM(order_items.price)         : R$ {price_total:>15,.2f}")
print(f"SUM(order_items.freight_value) : R$ {freight_total:>15,.2f}")
print(f"SUM(price + freight)           : R$ {(price_total + freight_total):>15,.2f}")
print(f"SUM(payments.payment_value)    : R$ {payment_total:>15,.2f}")
print()
print("Difference (payment - (price+freight)):",
      round(payment_total - (price_total + freight_total), 2))

SUM(order_items.price)         : R$   13,591,643.70
SUM(order_items.freight_value) : R$    2,251,909.54
SUM(price + freight)           : R$   15,843,553.24
SUM(payments.payment_value)    : R$   16,008,872.12

Difference (payment - (price+freight)): 165318.88


In [23]:
# Payment includes vouchers and discount codes — explains the small difference
print("Payment type breakdown:")
print(payments["payment_type"].value_counts())
print()
# Voucher rows: these reduce the effective revenue
voucher_value = payments[payments["payment_type"] == "voucher"]["payment_value"].sum()
print(f"Total voucher value: R$ {voucher_value:,.2f}")

Payment type breakdown:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Total voucher value: R$ 379,436.87


**Answer — Revenue Source:**

> **Revenue = `payment_value` from `order_payments`**, aggregated per `order_id`.
>
> Justification:
> - `price` excludes freight — understates what the customer paid.
> - `price + freight_value` is close but doesn't account for vouchers/discounts applied at checkout.
> - `payment_value` is the **actual amount the customer paid** (after discounts, including freight), making it the most accurate revenue metric.
>
> **Note:** For BQ #10 and #11 (payment method analysis), join `order_payments` to `order_items` via `order_id`. For orders with multiple payment rows (split payment), sum `payment_value` per `order_id` first.

## 9. Q4 — What are the Dimensions?

In [24]:
# DIM_CUSTOMER
print("DIM_CUSTOMER — from customers table")
print(customers.columns.tolist())
print(f"Rows: {customers.shape[0]:,} | Unique customer_unique_id: {customers['customer_unique_id'].nunique():,}")
print()

# DIM_PRODUCT
print("DIM_PRODUCT — from products + category_translation")
products_full = products.merge(category_translation, on="product_category_name", how="left")
print(products_full.columns.tolist())
print(f"Rows: {products_full.shape[0]:,} | Unique product_id: {products_full['product_id'].nunique():,}")
print()

# DIM_SELLER
print("DIM_SELLER — from sellers table")
print(sellers.columns.tolist())
print(f"Rows: {sellers.shape[0]:,} | Unique seller_id: {sellers['seller_id'].nunique():,}")
print()

# DIM_DATE
print("DIM_DATE — generated from order timestamps")
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
print(f"Date range: {orders['order_purchase_timestamp'].min().date()} → {orders['order_purchase_timestamp'].max().date()}")
print()

# DIM_LOCATION
print("DIM_LOCATION — from geolocation table (deduplicated by zip code)")
geo_dedup = geolocation.drop_duplicates(subset=["geolocation_zip_code_prefix"])
print(geo_dedup.columns.tolist())
print(f"Rows after dedup: {geo_dedup.shape[0]:,}")
print()

# DIM_PAYMENT
print("DIM_PAYMENT — from order_payments table")
print(payments["payment_type"].value_counts())

DIM_CUSTOMER — from customers table
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Rows: 99,441 | Unique customer_unique_id: 96,096

DIM_PRODUCT — from products + category_translation
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english']
Rows: 32,951 | Unique product_id: 32,951

DIM_SELLER — from sellers table
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
Rows: 3,095 | Unique seller_id: 3,095

DIM_DATE — generated from order timestamps
Date range: 2016-09-04 → 2018-10-17

DIM_LOCATION — from geolocation table (deduplicated by zip code)
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
Rows after dedup: 19,015

DIM_PAYMENT — from order_payments table
payment_type
credit_card  

**Answer — Dimensions (final schema):**

| Dimension | Source Table(s) | Primary Key | Notes |
|---|---|---|---|
| `DIM_CUSTOMER` | customers | `customer_unique_id` | Use `customer_unique_id`, NOT `customer_id` (per-order surrogate) |
| `DIM_PRODUCT` | products + category_translation | `product_id` | Merged English category name |
| `DIM_SELLER` | sellers | `seller_id` | `seller_state` supports BQ #8 |
| `DIM_DATE` | generated | `date_key` | year, quarter, month, weekday |
| `DIM_PAYMENT` | order_payments | `payment_key` | `payment_type`, `installment_group` |

> ⚠️ **`DIM_LOCATION` removed** from final schema — geolocation data is not required for any of the 17 BQs and deduplication logic adds ETL complexity without proportional value.
>
> ⚠️ **`customer_key`** in all 3 fact tables is looked up from `DIM_CUSTOMER` via `customer_unique_id` — never via `customer_id`.

## 10. Q5 — Do the data support all 17 Business Questions?

For each BQ, verify that the required columns exist in the source tables.

In [25]:
# BQ #1: Revenue over time
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["year_month"] = orders["order_purchase_timestamp"].dt.to_period("M")
monthly = (
    orders.merge(payments.groupby("order_id")["payment_value"].sum().reset_index(), on="order_id")
    .groupby("year_month")["payment_value"].sum()
)
print(f"BQ #1 — Revenue over time: {monthly.shape[0]} monthly periods available")
print(f"  Range: {monthly.index.min()} → {monthly.index.max()}")

BQ #1 — Revenue over time: 25 monthly periods available
  Range: 2016-09 → 2018-10


In [26]:
# BQ #2: Revenue by category
order_items_cat = (
    order_items
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
)
cat_revenue = order_items_cat.groupby("product_category_name_english")["price"].sum().sort_values(ascending=False)
print(f"BQ #2 — Revenue by category: {cat_revenue.shape[0]} categories")
print(cat_revenue.head(5))

BQ #2 — Revenue by category: 71 categories
product_category_name_english
health_beauty            1258681.34
watches_gifts            1205005.68
bed_bath_table           1036988.68
sports_leisure            988048.97
computers_accessories     911954.32
Name: price, dtype: float64


In [27]:
# BQ #3: AOV by region
order_total = payments.groupby("order_id")["payment_value"].sum().reset_index(name="order_value")
order_region = orders[["order_id", "customer_id"]].merge(
    customers[["customer_id", "customer_state"]], on="customer_id"
).merge(order_total, on="order_id")
aov_by_state = order_region.groupby("customer_state")["order_value"].mean().sort_values(ascending=False)
print(f"BQ #3 — AOV by state: {aov_by_state.shape[0]} states")
print(aov_by_state.head(5))

BQ #3 — AOV by state: 27 states
customer_state
PB    264.077836
AC    242.970617
RO    240.577866
AP    239.158824
AL    234.774964
Name: order_value, dtype: float64


In [28]:
# BQ #4: Customer Lifetime Value
# Must use customer_unique_id — the true person-level identifier.
# customer_id is a per-order surrogate; grouping by it gives CLV of 1 order only.
clv = (
    orders[["order_id", "customer_id"]]
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id"
    )
    .merge(
        payments.groupby("order_id")["payment_value"].sum().reset_index(),
        on="order_id"
    )
    .groupby("customer_unique_id")["payment_value"].sum()
    .sort_values(ascending=False)
)
print(f"BQ #4 — CLV: {clv.shape[0]:,} unique customers ranked by lifetime spend")
print(clv.head(5))

BQ #4 — CLV: 96,095 unique customers ranked by lifetime spend
customer_unique_id
0a0a92112bd4c708ca5fde585afaa872    13664.08
46450c74a0d8c5ca9395da1daac6c120     9553.02
da122df9eeddfedc1dc1f5349a1a690c     7571.63
763c8b1c9c68a0229c42c9fc6f662b93     7274.88
dc4802a71eae9be1dd28f5d788ceb526     6929.31
Name: payment_value, dtype: float64


In [29]:
# BQ #5: Repeat purchase rate
# NOTE: customer_id is unique per ORDER in Olist, not per person.
# Must join customers to get customer_unique_id (true person identifier)
# before counting repeat purchases.
orders_with_uid = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id"
)
customer_order_count = orders_with_uid.groupby("customer_unique_id")["order_id"].count()
repeat_customers = (customer_order_count > 1).sum()
total_customers   = customer_order_count.shape[0]
repeat_rate       = repeat_customers / total_customers * 100

print(f"BQ #5 — Repeat purchase rate: {repeat_rate:.2f}%")
print(f"  Repeat customers (≥2 orders) : {repeat_customers:,}")
print(f"  Total unique customers        : {total_customers:,}")
print()
print("Order count distribution:")
print(customer_order_count.value_counts().sort_index().head(10))

BQ #5 — Repeat purchase rate: 3.12%
  Repeat customers (≥2 orders) : 2,997
  Total unique customers        : 96,096

Order count distribution:
order_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64


In [30]:
# BQ #6: RFM — check column availability
# Same fix: group by customer_unique_id, not customer_id.
# customer_unique_id is the true person-level identifier in Olist.
snapshot_date = orders["order_purchase_timestamp"].max()

rfm_data = (
    orders[["order_id", "customer_id", "order_purchase_timestamp"]]
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id"
    )
    .merge(
        payments.groupby("order_id")["payment_value"].sum().reset_index(),
        on="order_id"
    )
    .groupby("customer_unique_id")
    .agg(
        Recency   = ("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
        Frequency = ("order_id", "count"),
        Monetary  = ("payment_value", "sum")
    )
)

print(f"BQ #6 — RFM base data: {rfm_data.shape[0]:,} unique customers")
print()
print(rfm_data.describe())

BQ #6 — RFM base data: 96,095 unique customers

            Recency     Frequency      Monetary
count  96095.000000  96095.000000  96095.000000
mean     287.730756      1.034809    166.594226
std      153.407846      0.214385    231.428912
min        0.000000      1.000000      0.000000
25%      163.000000      1.000000     63.120000
50%      268.000000      1.000000    108.000000
75%      397.000000      1.000000    183.530000
max      772.000000     17.000000  13664.080000


In [31]:
# BQ #7: Seller performance (revenue + review score)
seller_revenue = order_items.groupby("seller_id")["price"].sum().reset_index(name="revenue")
seller_reviews = (
    order_items[["order_id", "seller_id"]]
    .merge(reviews[["order_id", "review_score"]], on="order_id")
    .groupby("seller_id")["review_score"].mean().reset_index(name="avg_review")
)
seller_perf = seller_revenue.merge(seller_reviews, on="seller_id")
print(f"BQ #7 — Seller performance: {seller_perf.shape[0]:,} sellers with both revenue and review data")
print(seller_perf.sort_values("revenue", ascending=False).head(5))

BQ #7 — Seller performance: 3,090 sellers with both revenue and review data
                             seller_id    revenue  avg_review
854   4869f7a5dfa277a7dca6462dcf3b52b2  229472.63    4.122822
1010  53243585a1d6dc2643021fd1853d8905  222776.05    4.075980
878   4a3ca9315b744ce9f8e9374361493884  200472.92    3.803931
3019  fa1c13f2614d7b5c4749cbc52fecda94  194042.03    4.340206
1532  7c67e1448b00f6e969d365cea6b010ab  187923.89    3.348208


In [32]:
# BQ #8: Seller delivery delays
orders["order_delivered_customer_date"]  = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"]  = pd.to_datetime(orders["order_estimated_delivery_date"])
orders_delivered = orders.dropna(subset=["order_delivered_customer_date"])
orders_delivered = orders_delivered.copy()
orders_delivered["delay_days"] = (
    orders_delivered["order_delivered_customer_date"]
    - orders_delivered["order_estimated_delivery_date"]
).dt.days

seller_delay = (
    order_items[["order_id", "seller_id"]]
    .merge(orders_delivered[["order_id", "delay_days"]], on="order_id")
    .groupby("seller_id")["delay_days"].mean()
    .sort_values(ascending=False)
)
print(f"BQ #8 — Seller delivery delays: {seller_delay.shape[0]:,} sellers")
print(seller_delay.head(5))

BQ #8 — Seller delivery delays: 2,970 sellers
seller_id
df683dfda87bf71ac3fc63063fba369d    167.0
8e670472e453ba34a379331513d6aab1     35.0
8629a7efec1aab257e58cda559f03ba7     33.0
4fb41dff7c50136976d1a5cf004a42e2     33.0
eebb3372362aa9a46975164bed19a7e7     27.0
Name: delay_days, dtype: float64


In [33]:
# BQ #9: Revenue by customer state
state_revenue = (
    orders[["order_id", "customer_id"]]
    .merge(customers[["customer_id", "customer_state"]], on="customer_id")
    .merge(payments.groupby("order_id")["payment_value"].sum().reset_index(), on="order_id")
    .groupby("customer_state")["payment_value"].sum()
    .sort_values(ascending=False)
)
print(f"BQ #9 — Revenue by state: {state_revenue.shape[0]} states")
print(state_revenue.head(5))

BQ #9 — Revenue by state: 27 states
customer_state
SP    5998226.96
RJ    2144379.69
MG    1872257.26
RS     890898.54
PR     811156.38
Name: payment_value, dtype: float64


In [34]:
# BQ #10: Payment method distribution and AOV by payment type
pay_dist = payments["payment_type"].value_counts()
pay_aov = (
    payments.groupby(["order_id", "payment_type"])["payment_value"].sum()
    .reset_index()
    .groupby("payment_type")["payment_value"].mean()
    .sort_values(ascending=False)
)
print("BQ #10 — Payment method distribution:")
print(pay_dist)
print("\nAOV by payment type:")
print(pay_aov)

BQ #10 — Payment method distribution:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

AOV by payment type:
payment_type
credit_card    163.938098
boleto         145.034435
debit_card     142.663475
voucher         98.147147
not_defined      0.000000
Name: payment_value, dtype: float64


In [35]:
# BQ #11: Installment payment vs order value
install_aov = (
    payments[payments["payment_type"] == "credit_card"]
    .groupby("order_id")
    .agg(installments=("payment_installments", "max"),
         order_value=("payment_value", "sum"))
    .groupby("installments")["order_value"].mean()
)
print("BQ #11 — AOV by number of installments (credit card):")
print(install_aov)

BQ #11 — AOV by number of installments (credit card):
installments
0      94.315000
1      96.174939
2     127.637297
3     142.999558
4     164.286850
5     183.961783
6     210.535949
7     188.554143
8     309.142778
9     203.853214
10    417.092181
11    124.932174
12    321.678496
13    150.462500
14    167.962667
15    445.553108
16    292.694000
17    174.602500
18    486.483333
20    615.801765
21    243.700000
22    228.710000
23    236.480000
24    610.048889
Name: order_value, dtype: float64


In [36]:
# BQ #12 & #14: Delivery time and delay by state
orders_d = orders.dropna(subset=["order_delivered_customer_date"]).copy()
orders_d["delivery_days"] = (
    orders_d["order_delivered_customer_date"] - orders_d["order_purchase_timestamp"]
).dt.days
orders_d["delay_days"] = (
    orders_d["order_delivered_customer_date"] - orders_d["order_estimated_delivery_date"]
).dt.days

delivery_by_state = (
    orders_d[["order_id", "customer_id", "delivery_days", "delay_days"]]
    .merge(customers[["customer_id", "customer_state"]], on="customer_id")
    .groupby("customer_state")
    .agg(avg_delivery_days=("delivery_days", "mean"),
         avg_delay_days=("delay_days", "mean"))
    .sort_values("avg_delay_days", ascending=False)
)
print(f"BQ #12 & #14 — Delivery stats by state ({delivery_by_state.shape[0]} states):")
print(delivery_by_state.head(5))

BQ #12 & #14 — Delivery stats by state (27 states):
                avg_delivery_days  avg_delay_days
customer_state                                   
AL                      24.040302       -8.707809
MA                      21.117155       -9.571827
SE                      21.029851      -10.020896
ES                      15.331830      -10.496241
BA                      18.866400      -10.794533


In [37]:
# BQ #13: Late delivery rate (KPI)
orders_d["is_late"] = orders_d["delay_days"] > 0
late_rate = orders_d["is_late"].mean() * 100
on_time_rate = 100 - late_rate
print(f"BQ #13 — Late delivery rate  : {late_rate:.2f}%")
print(f"         On-time delivery rate: {on_time_rate:.2f}%")
print(f"         Total delivered orders: {orders_d.shape[0]:,}")

BQ #13 — Late delivery rate  : 6.77%
         On-time delivery rate: 93.23%
         Total delivered orders: 96,476


In [38]:
# BQ #15 & #16: Delivery performance vs review score
review_delivery = (
    orders_d[["order_id", "delivery_days", "delay_days", "is_late"]]
    .merge(reviews[["order_id", "review_score"]], on="order_id")
)
print(f"BQ #15 & #16 — Orders with both delivery data and review: {review_delivery.shape[0]:,}")
print("\nAvg review score — On-time vs Late:")
print(review_delivery.groupby("is_late")["review_score"].mean())
print()
print("Correlation (delivery_days vs review_score):",
      round(review_delivery["delivery_days"].corr(review_delivery["review_score"]), 4))

BQ #15 & #16 — Orders with both delivery data and review: 96,359

Avg review score — On-time vs Late:
is_late
False    4.289842
True     2.271139
Name: review_score, dtype: float64

Correlation (delivery_days vs review_score): -0.3337


In [39]:
# BQ #17: Category ratings
cat_ratings = (
    order_items[["order_id", "product_id"]]
    .merge(products[["product_id", "product_category_name"]], on="product_id")
    .merge(category_translation, on="product_category_name", how="left")
    .merge(reviews[["order_id", "review_score"]], on="order_id")
    .groupby("product_category_name_english")["review_score"].mean()
    .sort_values(ascending=False)
)
print(f"BQ #17 — Category ratings: {cat_ratings.shape[0]} categories")
print("Top 5 highest rated:")
print(cat_ratings.head(5))
print("\nTop 5 lowest rated:")
print(cat_ratings.tail(5))

BQ #17 — Category ratings: 71 categories
Top 5 highest rated:
product_category_name_english
cds_dvds_musicals            4.642857
fashion_childrens_clothes    4.500000
books_general_interest       4.446266
costruction_tools_tools      4.444444
flowers                      4.419355
Name: review_score, dtype: float64

Top 5 lowest rated:
product_category_name_english
fashion_male_clothing    3.641221
home_comfort_2           3.629630
office_furniture         3.493183
diapers_and_hygiene      3.256410
security_and_services    2.500000
Name: review_score, dtype: float64


## 11. Summary & Answers

### Q1 — Fact Table

Three fact tables, each at a different grain:

| Fact Table | Grain | Key Measures |
|---|---|---|
| `FACT_SALES` | 1 product in 1 order | price, freight_value |
| `FACT_PAYMENTS` | 1 payment record | payment_value, payment_installments |
| `FACT_ORDER_EXPERIENCE` | 1 order | delivery_days, delay_days, review_score |

> **Why 3 fact tables?** A single fact table with `payment_value` at order-item grain causes fan-out join — SUM(payment_value) double-counts when an order has multiple items. Separating grains eliminates this entirely.

---

### Q2 — Grain
> **FACT_SALES:** one row = one product line item in one order (`order_id` + `order_item_id`).
> Verified: `(order_id, order_item_id)` is a unique key with zero duplicates.

---

### Q3 — Revenue Source
> **Revenue = `SUM(payment_value)` from FACT_PAYMENTS**, grouped by `order_id`.
> - `price` alone excludes freight → understates actual payment.
> - `price + freight_value` does not account for vouchers/discounts applied at checkout.
> - `payment_value` = what the customer **actually paid** → correct revenue metric.

---

### Q4 — Dimensions

| Dimension | Source | Key Column | Notes |
|---|---|---|---|
| `DIM_CUSTOMER` | customers | `customer_unique_id` | ⚠️ NOT customer_id |
| `DIM_PRODUCT` | products + category_translation | `product_id` | English category name merged |
| `DIM_SELLER` | sellers | `seller_id` | seller_state for BQ #8 |
| `DIM_DATE` | generated | `date_key` | year / quarter / month / weekday |
| `DIM_PAYMENT` | order_payments | `payment_key` | payment_type, installment_group |

---

### Q5 — Business Question Coverage

| # | Business Question | Fact Table | Key Dimensions |
|---|---|---|---|
| 1 | Revenue over time | FACT_PAYMENTS | DIM_DATE |
| 2 | Revenue by category (growth/decline) | FACT_SALES | DIM_PRODUCT, DIM_DATE |
| 3 | AOV by region | FACT_PAYMENTS | DIM_CUSTOMER |
| 4 | Customer Lifetime Value | FACT_PAYMENTS | DIM_CUSTOMER |
| 5 | Repeat purchase rate | FACT_PAYMENTS | DIM_CUSTOMER |
| 6 | RFM segmentation | FACT_PAYMENTS | DIM_CUSTOMER, DIM_DATE |
| 7 | Seller performance (revenue + rating) | FACT_SALES + FACT_ORDER_EXPERIENCE | DIM_SELLER |
| 8 | Seller delivery delays | FACT_SALES + FACT_ORDER_EXPERIENCE | DIM_SELLER |
| 9 | Revenue by state | FACT_PAYMENTS | DIM_CUSTOMER |
| 10 | Payment method distribution & AOV | FACT_PAYMENTS | DIM_PAYMENT |
| 11 | Installments vs order value | FACT_PAYMENTS | DIM_PAYMENT |
| 12 | Delivery time by region | FACT_ORDER_EXPERIENCE | DIM_CUSTOMER |
| 13 | Late delivery rate (KPI) | FACT_ORDER_EXPERIENCE | — |
| 14 | Delivery delay by state | FACT_ORDER_EXPERIENCE | DIM_CUSTOMER |
| 15 | Delivery performance vs review score | FACT_ORDER_EXPERIENCE | — |
| 16 | Delayed orders → negative reviews | FACT_ORDER_EXPERIENCE | — |
| 17 | Category ratings | FACT_SALES + FACT_ORDER_EXPERIENCE | DIM_PRODUCT |

**All 17 Business Questions are supported by the Olist dataset.**

---

### Next Step
→ Design `create_dw.sql` using the 3 fact tables and 5 dimensions confirmed above.